# 02 Predictions And External Evaluation

Generated by `scripts/revision/create_revision_notebooks.py`.


## Setup

In [ ]:
from pathlib import Path
import json
import os
import subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive mount skipped or already mounted: {exc}')

DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
REPO_DIR = DRIVE_ROOT / 'ECG-RAMBA'
if not (REPO_DIR / 'configs' / 'config.py').exists():
    REPO_DIR = Path.cwd()

os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.chdir(REPO_DIR)

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)

def run(cmd, check=True):
    print(f'$ {cmd}')
    return subprocess.run(cmd, shell=True, check=check)


def run_script_if_exists(script_path, command):
    path = Path(script_path)
    if path.exists():
        run(command)
    else:
        print(f'SKIP: {script_path} is not implemented yet.')
        print(f'Planned command: {command}')


## Install Model Dependencies

In [ ]:
INSTALL_MODEL_DEPS = True
if INSTALL_MODEL_DEPS:
    !pip install -q mamba-ssm causal-conv1d>=1.2.0
else:
    print('Skipping mamba-ssm install. Ensure it is already available before prediction generation.')


## Prediction Contract

Every downstream metric script expects NPZ files with `y_true` and `y_prob`, both shaped `(N, C)`. Store prediction files under `reports/revision/predictions/`.

In [ ]:
from pathlib import Path
import numpy as np

pred_dir = Path('reports/revision/predictions')
pred_dir.mkdir(parents=True, exist_ok=True)

for path in sorted(pred_dir.glob('*.npz')):
    data = np.load(path, allow_pickle=True)
    keys = sorted(data.files)
    print(path, keys)
    if {'y_true', 'y_prob'}.issubset(keys):
        print('  y_true:', data['y_true'].shape, 'y_prob:', data['y_prob'].shape)


## Generate OOF Predictions

In [ ]:
RUN_HEAVY = False

command = (
    'python scripts/revision/01_generate_predictions.py '
    '--dataset oof --checkpoint-kind best'
)

if RUN_HEAVY:
    run(command)
else:
    print(f'Heavy run disabled. Set RUN_HEAVY=True to execute: {command}')


## External Prediction Commands

In [ ]:
print('PTB-XL and CPSC/Georgia prediction exporters are the next implementation step.')
print('Expected future outputs:')
print('- reports/revision/predictions/ptbxl_full_predictions.npz')
print('- reports/revision/predictions/cpsc_full_predictions.npz')


## Re-run Inventory

In [ ]:
!python scripts/revision/05_artifact_inventory.py
